In [8]:
// First cell initializes Spark Context, no need to create one in code

// Imports
import org.apache.spark.{SparkContext, SparkConf}
import scala.util.parsing.json._


import org.apache.spark.{SparkContext, SparkConf}
import scala.util.parsing.json._


In [21]:
// File locations
val USER = "e12233000"
val dev_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json"
val full_set_path = "hdfs:///user/dic24_shared/amazon-reviews/full/reviewscombined.json"
val stopwords_path = "hdfs:///user/"+USER+"/data/stopwords.txt"


USER: String = e12233000
dev_set_path: String = hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json
full_set_path: String = hdfs:///user/dic24_shared/amazon-reviews/full/reviewscombined.json
stopwords_path: String = hdfs:///user/e12233000/data/stopwords.txt


In [13]:
// Helper functions

// Print first k elements of a pair RDD, k default 2
def rdd_pprint(rdd: org.apache.spark.rdd.RDD[(String,String)], k: Int = 2): Unit = {
    rdd.take(k).foreach(println)
}

rdd_pprint: (rdd: org.apache.spark.rdd.RDD[(String, String)], k: Int)Unit


In [22]:
// Load devset into an RDD
val devset = sc.textFile(dev_set_path)

val stopwords = sc.textFile(stopwords_path).collect().toSet

devset: org.apache.spark.rdd.RDD[String] = hdfs:///user/dic24_shared/amazon-reviews/full/reviews_devset.json MapPartitionsRDD[14] at textFile at <console>:35
stopwords: scala.collection.immutable.Set[String] = Set(serious, latterly, absorbs, looks, particularly, used, e, printer, down, regarding, entirely, regardless, moreover, please, read, ourselves, able, behind, for, despite, s, maybe, viz, further, corresponding, x, any, wherein, across, name, allows, this, instead, in, taste, ought, myself, have, your, off, once, are, is, mon, his, oh, why, rd, knows, bulbs, too, among, course, greetings, somewhat, bibs, everyone, seen, likely, said, try, already, soon, nobody, got, given, song, using, less, am, consider, hence, than, n, accordingly, four, anyhow, want, three, forth, whereby, hims...


A little exploration of the Spark file structure:

In [31]:
val head = devset.take(2)
head.foreach(println)

{"reviewerID": "A2VNYWOPJ13AFP", "asin": "0981850006", "reviewerName": "Amazon Customer \"carringt0n\"", "helpful": [6, 7], "reviewText": "This was a gift for my other husband.  He's making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You'll love the food and it has provided us with an insight into the culture that produced it. It's all about broadening horizons.  Yum!!", "overall": 5.0, "summary": "Delish", "unixReviewTime": 1259798400, "reviewTime": "12 3, 2009", "category": "Patio_Lawn_and_Garde"}
{"reviewerID": "A2E5XXXC07AGA7", "asin": "B00002N66D", "reviewerName": "James", "helpful": [1, 1], "reviewText": "This is a very nice spreader.  It feels very solid and the pneumatic tires give it great maneuverability and handling over bumps.  Th

head: Array[String] = Array({"reviewerID": "A2VNYWOPJ13AFP", "asin": "0981850006", "reviewerName": "Amazon Customer \"carringt0n\"", "helpful": [6, 7], "reviewText": "This was a gift for my other husband.  He's making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You'll love the food and it has provided us with an insight into the culture that produced it. It's all about broadening horizons.  Yum!!", "overall": 5.0, "summary": "Delish", "unixReviewTime": 1259798400, "reviewTime": "12 3, 2009", "category": "Patio_Lawn_and_Garde"}, {"reviewerID": ...


In [9]:
val pairRDD = devset.map(line => {
  val json:Option[Any] = JSON.parseFull(line)
  val map:Map[String,Any] = json.get.asInstanceOf[Map[String, Any]]
  val category = map("category").toString
  val review = map("reviewText").toString
  (category, review)
})

pairRDD: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[4] at map at <console>:32


In [16]:
rdd_pprint(pairRDD,2)

(Patio_Lawn_and_Garde,This was a gift for my other husband.  He's making us things from it all the time and we love the food.  Directions are simple, easy to read and interpret, and fun to make.  We all love different kinds of cuisine and Raichlen provides recipes from everywhere along the barbecue trail as he calls it. Get it and just open a page.  Have at it.  You'll love the food and it has provided us with an insight into the culture that produced it. It's all about broadening horizons.  Yum!!)
(Patio_Lawn_and_Garde,This is a very nice spreader.  It feels very solid and the pneumatic tires give it great maneuverability and handling over bumps.  The control arm is solid metal, not a cable, which gives you precise control and will last a long time.  The settings take some experimentation with your various products to get it right, but that is true of any spreader.  It has good distribution... probably flings material a little farther on the right side than the left, but it is far mor

In [27]:
var wordCategoryRDD = pairRDD.flatMap{ case (category, text) =>
    val words = text.toLowerCase.split("[ \t0-9\\(\\)\\[\\]\\{\\}\\.\\!\\?,;:\\+=\\-_\"'`~#@&%*%€$§\\\\/|<>^]+")
    words.map(word => (category, word))
}

wordCategoryRDD = wordCategoryRDD.filter{ case (category, word) => !stopwords.contains(word) }

wordCategoryRDD: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[20] at filter at <console>:39
wordCategoryRDD: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[20] at filter at <console>:39


In [30]:
rdd_pprint(wordCategoryRDD,20)

(Patio_Lawn_and_Garde,gift)
(Patio_Lawn_and_Garde,husband)
(Patio_Lawn_and_Garde,making)
(Patio_Lawn_and_Garde,things)
(Patio_Lawn_and_Garde,time)
(Patio_Lawn_and_Garde,love)
(Patio_Lawn_and_Garde,food)
(Patio_Lawn_and_Garde,directions)
(Patio_Lawn_and_Garde,simple)
(Patio_Lawn_and_Garde,easy)
(Patio_Lawn_and_Garde,interpret)
(Patio_Lawn_and_Garde,make)
(Patio_Lawn_and_Garde,love)
(Patio_Lawn_and_Garde,kinds)
(Patio_Lawn_and_Garde,cuisine)
(Patio_Lawn_and_Garde,raichlen)
(Patio_Lawn_and_Garde,recipes)
(Patio_Lawn_and_Garde,barbecue)
(Patio_Lawn_and_Garde,trail)
(Patio_Lawn_and_Garde,calls)


In [ ]:
# Terminate Spark context
sc.stop()